# RNN, LSTM & GRU Model Development

## Dataset Overview: New York City Taxi Data

This notebook analyzes a New York City taxi dataset. The dataset contains records of taxi trips, providing insights into various aspects of each journey. Below is a detailed description of the key features:

*   **id**: A unique identifier for each trip.
*   **vendor_id**: An identifier indicating the taxi vendor (e.g., Creative Mobile Technologies, VeriFone Inc.).
*   **pickup_datetime**: The date and time when the taxi trip started.
*   **dropoff_datetime**: The date and time when the taxi trip ended.
*   **passenger_count**: The number of passengers in the taxi for the trip.
*   **pickup_longitude**: The longitude coordinate of the pickup location.
*   **pickup_latitude**: The latitude coordinate of the pickup location.
*   **dropoff_longitude**: The longitude coordinate of the dropoff location.
*   **dropoff_latitude**: The latitude coordinate of the dropoff location.
*   **store_and_fwd_flag**: A flag indicating whether the trip record was 'stored and forwarded' (Y) or not (N). This means the data was stored in the vehicle's memory before being sent to the server, typically if there was no network connection.

The primary goal of this analysis is to demonstrate the application of Recurrent Neural Networks (RNN), Long Short-Term Memory (LSTM), and Gated Recurrent Unit (GRU) models for time series prediction, specifically focusing on the `passenger_count`.

### Step 1: Importing Libraries

This section imports all the necessary libraries for data manipulation, numerical operations, plotting, data scaling, splitting, and building deep learning models with TensorFlow/Keras, as well as evaluation metrics.

```
import pandas as pd # Used for data manipulation and analysis
import numpy as np # Used for numerical operations on arrays
import matplotlib.pyplot as plt # Used for plotting graphs and visualizations
from sklearn.preprocessing import MinMaxScaler # Used for normalizing data ranges
from sklearn.model_selection import train_test_split # Used for splitting data into train/test sets
import tensorflow as tf # Core framework for deep learning
from tensorflow import keras # High-level API for building neural networks
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score # Metrics for model evaluation
```

### Step 2: Data Preparation

This section focuses on loading the dataset and performing initial inspections to understand its structure and content.

```
# Read the NYC Taxi dataset from a CSV file
df = pd.read_csv("ny_taxi_data.csv")
```

```
# View the first 5 rows to understand the data structure and column names
display(df.head())
```

```
# Select only the 'passenger_count' column values for our time series analysis
data = df["passenger_count"].values
```

To prepare the data for neural networks, it's essential to scale the values. The `MinMaxScaler` transforms features by scaling them to a given range, typically between 0 and 1. This helps in optimizing model performance.

```
# Initialize the MinMaxScaler to scale values between 0 and 1
scaler = MinMaxScaler()
```

```
# Reshape data to 2D array as required by the scaler's fit_transform method
data = data.reshape(-1, 1)
```

```
# Normalize the passenger count data to help the neural network converge faster
data = scaler.fit_transform(data)
```

For time series forecasting, we need to create sequences of data, where each sequence represents a window of past observations used to predict the next value. This involves defining a `sequence_length` which determines how many previous time steps are considered.

```
# Define how many previous time steps (window size) to use for predicting the next value
sequence_length = 5
```

```
sequences = []
targets = []
```

```
# Iterate through the data to create sliding window sequences

for i in range(len(data) - sequence_length):

    # Extract a sequence of length 'sequence_length'
    sequences.append(data[i:i + sequence_length])

    # The target is the very next value immediately following the sequence
    targets.append(data[i + sequence_length])
```

```
# Convert lists to numpy arrays for compatibility with TensorFlow models
sequences = np.array(sequences)
targets = np.array(targets)
```

To evaluate the model's performance on unseen data, the dataset is split into training and testing sets. The training set is used to teach the model, while the testing set provides an unbiased evaluation of its accuracy.

```
# Split the sequence data into 80% training data and 20% testing data
X_train, X_test, y_train, y_test = train_test_split(sequences, targets, test_size=0.2, random_state=42)
```

```
# Print the shape of the training set to verify the samples, time steps, and features
print(f"Training set shape: {X_train.shape}")
```

### Step 4: Build and Train RNN Model

Recurrent Neural Networks (RNNs) are a class of neural networks designed to recognize patterns in sequences of data. They process sequences by maintaining an internal state (memory) that captures information about previous elements in the sequence.

### Step 5: Model Evaluation Utilities
We define a reusable function to predict values on the test set, inverse the scaling back to original taxi passenger counts, and calculate standard regression metrics (MSE, RMSE, MAE, R2).

### Step 4: Build and Train RNN Model

```
# Construct a Simple Recurrent Neural Network (RNN) model
model_rnn = keras.Sequential([

    # Input layer matching the defined sequence length
    keras.layers.Input(shape=(X_train.shape[1], 1)),

    # RNN layer with 100 units and ReLU activation for capturing sequence patterns
    keras.layers.SimpleRNN(100, activation='relu'),

    # Output layer for regression predicting a single continuous value
    keras.layers.Dense(1)

], name='RNN')
```

```
model_rnn.summary()
```

```
# Compile the model using Adam optimizer and Mean Squared Error loss
model_rnn.compile(optimizer='adam', loss='mean_squared_error', metrics=['accuracy'])
```

```
# Train the RNN model for 10 epochs using 20% of the training data for internal validation
history_rnn = model_rnn.fit(X_train, y_train, epochs=100, batch_size=32, verbose=1, validation_split=0.2)
```

### RNN Model Evaluation

After training, we evaluate the RNN model's performance on unseen test data. The `evaluate_model` function calculates key regression metrics such as Mean Squared Error (MSE), Root Mean Squared Error (RMSE), Mean Absolute Error (MAE), and R-squared (R2).

```
# Define a function to evaluate model performance on test data across multiple metrics

def evaluate_model(model, X_test, y_test, scaler):

    # Reshape test features for model prediction input
    X_test_reshaped = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

    # Generate predictions and reverse scaling to get original units (passenger counts)
    y_pred = model.predict(X_test_reshaped)
    y_pred = scaler.inverse_transform(y_pred).flatten()

    # Reverse scaling for actual test ground truth values
    y_test_inv = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

    # Calculate key regression performance metrics
    mse = mean_squared_error(y_test_inv, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test_inv, y_pred)
    r2 = r2_score(y_test_inv, y_pred)
    return {'mse': mse, 'rmse': rmse, 'mae': mae, 'r2': r2}
```

```
# Calculate and store performance metrics for the Simple RNN model
metrics_rnn = evaluate_model(model_rnn, X_test, y_test, scaler)
```

```
# Print RNN model's evaluation metrics.
print(f"RNN Metrics: MSE={metrics_rnn['mse']:.4f}, " \
      f"RMSE={metrics_rnn['rmse']:.4f}, MAE={metrics_rnn['mae']:.4f}, " \
      f"R2={metrics_rnn['r2']:.4f}")
```

### RNN Model Training and Validation Loss

Visualizing the training and validation loss over epochs helps in understanding the learning process of the RNN model. A decreasing loss indicates that the model is learning to make better predictions, while comparing training and validation loss helps detect overfitting.

```
# Plot the loss curves to check for learning progress and potential overfitting
plt.figure(figsize=(10, 5))
plt.plot(history_rnn.history['loss'], label='RNN Training Loss')
plt.plot(history_rnn.history['val_loss'], label='RNN Validation Loss')
plt.title('RNN Model Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.show()
```

The model is trained using the `fit` method on the training data. The `epochs` parameter defines the number of complete passes through the training dataset, and `batch_size` is the number of samples per gradient update. `verbose=0` suppresses output during training.

```
# Visualize the Training and Validation Accuracy history for the RNN
plt.figure(figsize=(10, 5))
plt.plot(history_rnn.history['accuracy'], label='RNN Training Accuracy')
plt.plot(history_rnn.history['val_accuracy'], label='RNN Validation Accuracy')
plt.title('RNN Model Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()
```

### Step 5: Build and Train LSTM Model

Long Short-Term Memory (LSTM) networks are a special kind of RNN, capable of learning long-term dependencies. They are widely used for sequence prediction tasks due to their ability to mitigate the vanishing gradient problem inherent in simple RNNs.

Similar to the RNN, the LSTM model is constructed sequentially. It uses an `LSTM` layer followed by a `Dense` output layer. The compilation uses the same Adam optimizer and MSE loss.

```
# Construct an Long Short-Term Memory (LSTM) model to capture long-term dependencies
model_lstm = keras.Sequential([
    keras.layers.Input(shape=(X_train.shape[1], 1)),
    keras.layers.LSTM(100, activation='relu'),
    keras.layers.Dense(1)
], name='LSTM')
```

```
model_lstm.summary()
```

```
# Compile the LSTM model with identical settings for fair comparison
model_lstm.compile(optimizer='adam', loss='mean_squared_error', metrics=['accuracy'])
```

```
# Train the LSTM model and store its training history
history_lstm = model_lstm.fit(X_train, y_train, epochs=100, batch_size=32, verbose=1, validation_split=0.2)
```

```
# Calculate and store performance metrics for the LSTM model
metrics_lstm = evaluate_model(model_lstm, X_test, y_test, scaler)
```

```
# Print RNN model's evaluation metrics.
print(f"RNN Metrics: MSE={metrics_lstm['mse']:.4f}, " \
      f"RMSE={metrics_lstm['rmse']:.4f}, MAE={metrics_lstm['mae']:.4f}, " \
      f"R2={metrics_lstm['r2']:.4f}")
```

```
# Plot the training and validation loss for the LSTM model to monitor convergence
plt.figure(figsize=(10, 5))
plt.plot(history_lstm.history['loss'], label='LSTM Training Loss')
plt.plot(history_lstm.history['val_loss'], label='LSTM Validation Loss')
plt.title('LSTM Model Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.show()
```

```
plt.figure(figsize=(10, 5))
plt.plot(history_lstm.history['accuracy'], label='LSTM Training Accuracy')
plt.plot(history_lstm.history['val_accuracy'], label='LSTM Validation Accuracy')
plt.title('LSTM Model Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()
```

### Step 6: Build and Train GRU Model

### Step 6: Build and Train GRU Model

Gated Recurrent Units (GRUs) are another type of recurrent neural network, often considered a simplified version of LSTMs. They also address the vanishing gradient problem and are known for their efficiency and good performance in many sequence learning tasks.

The GRU model follows the same sequential structure, incorporating a `GRU` layer and a `Dense` output layer. It is compiled with the Adam optimizer and Mean Squared Error loss, ensuring a consistent evaluation setup across models.

```
# Construct a Gated Recurrent Unit (GRU) model, a modern and efficient variant of RNN
model_gru = keras.Sequential([
    keras.layers.Input(shape=(X_train.shape[1], 1)),
    keras.layers.GRU(100, activation='relu'),
    keras.layers.Dense(1)
], name='GRU')
```

```
model_gru.summary()
```

```
# Compile the GRU model
model_gru.compile(optimizer='adam', loss='mean_squared_error', metrics=['accuracy'])
```

```
# Train the GRU model and store its training history
history_gru = model_gru.fit(X_train, y_train, epochs=100, batch_size=32, verbose=1, validation_split=0.2)
```

```
# Calculate and store performance metrics for the GRU model
metrics_gru = evaluate_model(model_gru, X_test, y_test, scaler)
```

```
# Print RNN model's evaluation metrics.
print(f"RNN Metrics: MSE={metrics_gru['mse']:.4f}, " \
      f"RMSE={metrics_gru['rmse']:.4f}, MAE={metrics_gru['mae']:.4f}, " \
      f"R2={metrics_gru['r2']:.4f}")
```

```
# Plot the training and validation loss for the GRU model
plt.figure(figsize=(10, 5))
plt.plot(history_gru.history['loss'], label='GRU Training Loss')
plt.plot(history_gru.history['val_loss'], label='GRU Validation Loss')
plt.title('GRU Model Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.show()
```

```
plt.figure(figsize=(10, 5))
plt.plot(history_gru.history['accuracy'], label='GRU Training Accuracy')
plt.plot(history_gru.history['val_accuracy'], label='GRU Validation Accuracy')
plt.title('GRU Model Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()
```

## Step 7: Model Comparison & Conclusion

In this final step, we aggregate the evaluation metrics from all three models (RNN, LSTM, and GRU) into a single DataFrame. This allows for a direct comparison of their performance on the New York City taxi passenger count prediction task.

```
# Consolidate performance metrics from all three model types into a single dictionary
comparison_data = {
    'Model': ['Simple RNN', 'LSTM', 'GRU'],
    'MSE': [metrics_rnn['mse'], metrics_lstm['mse'], metrics_gru['mse']],
    'RMSE': [metrics_rnn['rmse'], metrics_lstm['rmse'], metrics_gru['rmse']],
    'MAE': [metrics_rnn['mae'], metrics_lstm['mae'], metrics_gru['mae']],
    'R2 Score': [metrics_rnn['r2'], metrics_lstm['r2'], metrics_gru['r2']]
}

# Create a Pandas DataFrame to display the comparison table clearly
comparison_df = pd.DataFrame(comparison_data)
# Display the results rounded to 4 decimal places for better readability
display(comparison_df.round(4))
```

### Final Observations

1. **Performance**: Compare the Mean Squared Error (MSE) and R2 Score across models. Typically, LSTM and GRU outperform Simple RNNs in time-series tasks due to their ability to handle long-term dependencies.
2. **Training Stability**: Review the loss curves generated in the previous steps. A smoother validation loss curve usually indicates a more stable and reliable model.
3. **Recommendation**: Based on the metrics above, the model with the lowest RMSE and highest R2 Score is generally preferred for deployment.

## Summary of Results

In this notebook, we successfully:
1.  **Prepared** NYC Taxi data for time-series forecasting.
2.  **Developed** three distinct architectures: Simple RNN, LSTM, and GRU.
3.  **Evaluated** and compared them using regression metrics.

This provides a foundation for more complex time-series forecasting tasks.

```

```